# Filtering rounding with NumPy and JAX

This self-contained companion demonstrates arithmetic boundaries with small,
generated inputs. It requires **NumPy and JAX**, uses JAX's ordinary default
device, and runs from a clean namespace. No frame files, waveform library or
PyCBC installation is required.

These are arithmetic demonstrations. NumPy is the host arithmetic control;
it is not a substitute for the original PyCBC/LAL pipeline. The
[PyCBC comparison notebook](jax_filtering_numerical_differences.ipynb)
separately validates original implementation routes, dtype, shape, bytes and
series metadata. Displayed results apply to the recorded libraries/device.


In [1]:
import platform
import numpy as np
import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp

DEVICE = jax.devices()[0]
print({'Python': platform.python_version(), 'NumPy': np.__version__,
       'JAX': jax.__version__, 'device': DEVICE.platform,
       'hardware': DEVICE.device_kind, 'x64': jax.config.jax_enable_x64})


def exact_bytes(left, right):
    left, right = np.asarray(left), np.asarray(right)
    return (left.shape == right.shape and left.dtype == right.dtype
            and left.tobytes() == right.tobytes())


def differing_values(left, right):
    return int(np.count_nonzero(np.asarray(left) != np.asarray(right)))


def scalar_table(title, rows):
    print(title)
    for name, value in rows:
        print(f'  {name:<28} {float(value):.17g}')


{'Python': '3.13.3', 'NumPy': '2.2.5', 'JAX': '0.11.2', 'device': 'gpu', 'hardware': 'NVIDIA GeForce RTX 3090', 'x64': True}


## Same affine updates, different grouping

A state update is `state[n] = a[n]*state[n-1] + b[n]`. Sequential execution
and a parallel prefix scan compose the same affine maps, but addition and
multiplication are not associative in floating point. Unit coefficients
remove coefficient construction and product rounding from this example.


In [2]:
def sequential_affine(coefficients, offsets):
    state = np.float64(0)
    output = np.empty_like(offsets)
    for index, (coefficient, offset) in enumerate(zip(coefficients, offsets)):
        state = np.float64(coefficient*state + offset)
        output[index] = state
    return output


@jax.jit
def parallel_affine(coefficients, offsets):
    def compose(left, right):
        return right[0]*left[0], right[0]*left[1]+right[1]
    return jax.lax.associative_scan(compose, (coefficients, offsets))[1]


affine_a = np.ones(4, dtype=np.float64)
affine_b = np.array([1e16, 1, -1e16, 1], dtype=np.float64)
sequential_state = sequential_affine(affine_a, affine_b)
with jax.default_device(DEVICE):
    parallel_state = np.asarray(parallel_affine(
        jnp.asarray(affine_a), jnp.asarray(affine_b)))
assert sequential_state[-1] == 1 and parallel_state[-1] == 0
assert sequential_state.dtype == parallel_state.dtype == np.float64
print('Sequential states:', sequential_state)
print('Parallel states:  ', parallel_state)
print('Same maps and zero initial state; final values: 1 versus 0')


Sequential states: [1.e+16 1.e+16 0.e+00 1.e+00]
Parallel states:   [1.e+16 1.e+16 0.e+00 0.e+00]
Same maps and zero initial state; final values: 1 versus 0


## Change coefficient inputs separately

The next example keeps the sequential algorithm and offsets fixed, changing
only each coefficient by one representable float64 step. This is a constructed
coefficient perturbation, not a claim that NumPy coefficients equal LAL's or
that a particular coefficient library produces this perturbation.


In [3]:
coefficient_a = np.full(64, .9, dtype=np.float64)
coefficient_b = np.random.default_rng(8).normal(size=64).astype(np.float64)
coefficient_changed = np.nextafter(coefficient_a, np.float64(np.inf))
coefficient_original_state = sequential_affine(coefficient_a, coefficient_b)
coefficient_changed_state = sequential_affine(coefficient_changed, coefficient_b)
coefficient_delta = coefficient_changed_state-coefficient_original_state
assert np.max(np.abs(coefficient_delta)) > 0
scalar_table('Fixed sequential recurrence; one-step coefficient change', [
    ('coefficient step', coefficient_changed[0]-coefficient_a[0]),
    ('original final state', coefficient_original_state[-1]),
    ('changed final state', coefficient_changed_state[-1]),
    ('maximum state difference', np.max(np.abs(coefficient_delta)))])


Fixed sequential recurrence; one-step coefficient change
  coefficient step             1.1102230246251565e-16
  original final state         1.4857483569304053
  changed final state          1.4857483569304051
  maximum state difference     3.9968028886505635e-15


## Freeze reciprocals before complex multiplication

For a positive real denominator, compare complex division, componentwise
division and multiplication by a previously rounded float32 reciprocal.
Their real-arithmetic formulas agree; their rounding boundaries need not.
CPU and device reciprocal evaluations are measured independently.

The explicit frozen-CPU-reciprocal control must reproduce NumPy's complex
quotient for these generated values. The device quotient's equality with its
own frozen reciprocal is **reported**, not assumed for every device/library.
No particular vendor instruction is inferred from a matching control.


In [4]:
reciprocal_rng = np.random.default_rng(8)
reciprocal_values = (reciprocal_rng.normal(size=65)
                     + 1j*reciprocal_rng.normal(size=65)).astype(np.complex64)
reciprocal_psd = np.geomspace(.01, 10, 65).astype(np.float32)
with np.errstate(divide='raise', invalid='raise'):
    cpu_quotient = reciprocal_values/reciprocal_psd
    cpu_reciprocal = np.float32(1)/reciprocal_psd
    cpu_frozen_quotient = reciprocal_values*cpu_reciprocal
assert exact_bytes(cpu_quotient, cpu_frozen_quotient)


@jax.jit
def device_division(values, denominators):
    return values/denominators


@jax.jit
def device_reciprocal(denominators):
    return jnp.float32(1)/denominators


@jax.jit
def frozen_reciprocal_product(values, reciprocals):
    return values*reciprocals


@jax.jit
def component_division(values, denominators):
    return jax.lax.complex(values.real/denominators,
                           values.imag/denominators)


with jax.default_device(DEVICE):
    reciprocal_z, reciprocal_d = map(jnp.asarray,
                                     (reciprocal_values, reciprocal_psd))
    device_quotient = np.asarray(device_division(reciprocal_z, reciprocal_d))
    device_inverse = np.asarray(device_reciprocal(reciprocal_d))
    device_components = np.asarray(component_division(reciprocal_z, reciprocal_d))
    frozen_cpu = np.asarray(frozen_reciprocal_product(
        reciprocal_z, jnp.asarray(cpu_reciprocal)))
    frozen_device = np.asarray(frozen_reciprocal_product(
        reciprocal_z, jnp.asarray(device_inverse)))
assert exact_bytes(frozen_cpu, cpu_quotient)
print('Differing values among 65 generated inputs:')
for label, left, right in [
    ('CPU versus device reciprocal', cpu_reciprocal, device_inverse),
    ('CPU versus device quotient', cpu_quotient, device_quotient),
    ('device quotient versus components', device_quotient, device_components),
    ('device quotient versus frozen device', device_quotient, frozen_device),
    ('CPU quotient versus frozen CPU', cpu_quotient, frozen_cpu)]:
    print(f'  {label:<39} {differing_values(left, right):2d}; '
          f'dtype/shape/bytes exact: {exact_bytes(left, right)}')


Differing values among 65 generated inputs:
  CPU versus device reciprocal             3; dtype/shape/bytes exact: False
  CPU versus device quotient               3; dtype/shape/bytes exact: False
  device quotient versus components        0; dtype/shape/bytes exact: True
  device quotient versus frozen device     0; dtype/shape/bytes exact: True
  CPU quotient versus frozen CPU           0; dtype/shape/bytes exact: True


## Point-sum additions: two changes can partly cancel

Both groupings below use the three products
`a = vr*(pr+pi)`, `b = pr*(vi-vr)`, `c = pi*(vr+vi)`.
Sample zero makes every phase `(1, 0)`, so only addition grouping changes.
The per-frequency carry is `R+(a-c), I+(a+b)`.
An observed scalar CPU path groups two frequencies as
`((a1-c1)+(a0-c0))+R, ((I+b0)+(b1+a0))+a1`, then uses
`(b+I)+a` for its scalar remainder.

This float32 NumPy model demonstrates those groupings; it does not assume
every compiler or candidate geometry emits that CPU path. First change only
the recurrence, then change bin-power accumulation from `S+(R*R+I*I)` to
`(S+R*R)+I*I`. The complete original-routine validation is in the companion
PyCBC notebook.


In [5]:
point_parts = np.random.default_rng(18).normal(size=(2, 3, 2)).astype(np.float32)
point_a = point_parts[:, :, 0].copy()
point_b = point_parts[:, :, 1]-point_a
point_c = np.zeros_like(point_a)
per_frequency = np.zeros((2, 2), dtype=np.float32)
for frequency in range(3):
    per_frequency[:, 0] += point_a[:, frequency]-point_c[:, frequency]
    per_frequency[:, 1] += point_a[:, frequency]+point_b[:, frequency]
paired = np.zeros_like(per_frequency)
paired[:, 0] = ((point_a[:, 1]-point_c[:, 1])
               + (point_a[:, 0]-point_c[:, 0]))+paired[:, 0]
paired[:, 1] = ((paired[:, 1]+point_b[:, 0])
               + (point_b[:, 1]+point_a[:, 0]))+point_a[:, 1]
paired[:, 0] = (point_a[:, 2]-point_c[:, 2])+paired[:, 0]
paired[:, 1] = (point_b[:, 2]+paired[:, 1])+point_a[:, 2]


def accumulated_power(sums, interleave=False):
    total = np.float32(0)
    for real, imag in sums:
        real_square, imag_square = real*real, imag*imag
        total = ((total+real_square)+imag_square if interleave
                 else total+(real_square+imag_square))
    return total


point_source = accumulated_power(per_frequency)
point_paired = accumulated_power(paired)
point_interleaved = accumulated_power(paired, interleave=True)
point_step = np.float64(np.spacing(point_source))
assert (np.float64(point_paired)-point_source)/point_step == 2
assert (np.float64(point_interleaved)-point_paired)/point_step == -1
scalar_table('Same six generated samples; point power', [
    ('per-frequency grouping', point_source),
    ('paired recurrence only', point_paired),
    ('paired and interleaved total', point_interleaved),
    ('one float32 step', point_step)])
print('Recurrence adds two steps; final accumulation removes one.')


Same six generated samples; point power
  per-frequency grouping       26.898221969604492
  paired recurrence only       26.898225784301758
  paired and interleaved total 26.898223876953125
  one float32 step             1.9073486328125e-06
Recurrence adds two steps; final accumulation removes one.


## Coherent subtraction amplifies preceding rounding

At sample zero, the unnormalized inverse-transform SNR is the sum of the
**same correlation** used to calculate the bin powers. No independently
invented SNR is supplied. In exact arithmetic the normalized statistic is
`norm² * (p*S - |raw_snr|²)`; this example sets `norm=1`.

The two large terms nearly cancel. Advancing either the held bin-power sum or
one raw SNR component by one float32 step demonstrates sensitivity; it does
not claim either perturbation is produced by every compiled kernel.


In [6]:
coherent_edges = np.array([1, 17, 33, 49, 65])
coherent_correlation = np.zeros(256, dtype=np.complex64)
coherent_responses = np.array([44.75+41.5j, 38.25+41.5j]*2, dtype=np.complex64)
for start, stop, response in zip(coherent_edges[:-1], coherent_edges[1:],
                                 coherent_responses):
    coherent_correlation[start:stop] = response/(stop-start)
coherent_bin_sums = np.asarray([
    np.sum(coherent_correlation[start:stop], dtype=np.complex64)
    for start, stop in zip(coherent_edges[:-1], coherent_edges[1:])])
coherent_s = accumulated_power(np.stack(
    (coherent_bin_sums.real, coherent_bin_sums.imag), axis=1))
coherent_raw = np.asarray([np.sum(coherent_correlation,
                                  dtype=np.complex64)], dtype=np.complex64)


def coherent_finish(bin_power, raw_snr):
    return np.float32(4)*bin_power-(raw_snr.conj()*raw_snr).real


coherent_a = np.float32(4)*coherent_s
coherent_b = (coherent_raw.conj()*coherent_raw).real
coherent_chi = coherent_finish(coherent_s, coherent_raw)
coherent_s_next = np.nextafter(coherent_s, np.float32(np.inf))
coherent_raw_next = coherent_raw.copy()
coherent_raw_next.real = np.nextafter(coherent_raw_next.real, np.float32(np.inf))
coherent_from_s = coherent_finish(coherent_s_next, coherent_raw)
coherent_from_raw = coherent_finish(coherent_s, coherent_raw_next)
assert coherent_a == 55281 and coherent_b[0] == 55112 and coherent_chi[0] == 169
assert (coherent_from_s-coherent_chi)[0] == .00390625
assert (coherent_from_raw-coherent_chi)[0] == -.00390625
coherent_relative_s = (np.float64(coherent_s_next)-coherent_s)/coherent_s
coherent_amplification = ((np.float64(coherent_from_s[0])-coherent_chi[0])
                          /coherent_chi[0]/coherent_relative_s)
with jax.default_device(DEVICE):
    device_coherent = np.asarray(jax.jit(
        lambda power, raw: 4*power-jnp.real(raw*jnp.conj(raw)))(
            jnp.asarray(coherent_s), jnp.asarray(coherent_raw)))
assert exact_bytes(coherent_chi, device_coherent)
scalar_table('Same-correlation coherent example', [
    ('A = p*S', coherent_a), ('B = |raw_snr|²', coherent_b[0]),
    ('A-B', coherent_chi[0]), ('one-step S result', coherent_from_s[0]),
    ('one-step SNR result', coherent_from_raw[0]),
    ('relative S amplification', coherent_amplification)])
print('Raw SNR derived from the correlation:', coherent_raw)


Same-correlation coherent example
  A = p*S                      55281
  B = |raw_snr|²               55112
  A-B                          169
  one-step S result            169.00390625
  one-step SNR result          168.99609375
  relative S amplification     327.10650887573962
Raw SNR derived from the correlation: [166.+166.j]


## Final complex64 SNR and Python scalar precision

A binary64 normalization scalar can lose its extra bits when combined with a
complex64 array. Here two distinct Python floats become the same float32
scalar, while a larger step changes the complex components. Magnitude
rounding is displayed separately; no universal CPU/device magnitude equality
is assumed.


In [7]:
snr_raw = np.array([1+2j], dtype=np.complex64)
snr_scalars = [1., float(np.nextafter(np.float64(1), np.float64(np.inf))),
               float(np.nextafter(np.float32(1), np.float32(np.inf)))]
snr_host = [snr_raw*scalar for scalar in snr_scalars]
with jax.default_device(DEVICE):
    snr_device = [np.asarray(jax.jit(lambda values, scalar: values*scalar)(
        jnp.asarray(snr_raw), scalar)) for scalar in snr_scalars]
assert snr_scalars[0] != snr_scalars[1]
assert exact_bytes(snr_host[0], snr_host[1])
assert not exact_bytes(snr_host[0], snr_host[2])
for host, device in zip(snr_host, snr_device):
    assert host.dtype == device.dtype == np.complex64
    assert exact_bytes(host, device)
print('scalar hex                 float32 scalar   scaled complex   magnitude')
for scalar, scaled in zip(snr_scalars, snr_host):
    print(f'{scalar.hex():<26} {np.float32(scalar)!s:<16} '
          f'{scaled[0]!s:<17} {np.abs(scaled)[0]}')
print('First two binary64 scalars collapse at the complex64 multiplication.')


scalar hex                 float32 scalar   scaled complex   magnitude
0x1.0000000000000p+0       1.0              (1+2j)            2.2360680103302
0x1.0000000000001p+0       1.0              (1+2j)            2.2360680103302
0x1.0000020000000p+0       1.0000001        (1.0000001+2.0000002j) 2.2360682487487793
First two binary64 scalars collapse at the complex64 multiplication.


## Validation scope

Assertions check the generated controls and their stated arithmetic
identities. They do not validate original pipeline behavior, scientific
trigger output, waveform generation or opaque FFT-library internals. Use the
[PyCBC/LAL comparison notebook](jax_filtering_numerical_differences.ipynb)
for original-route validation. The default production path remains on device;
this companion measures arithmetic mechanisms independently.
